# Modélisation - comparaison des modèles

**Objectif :** comparer plusieurs modèles et jeux de variables, puis justifier le choix
du modèle final et de son seuil de décision.

**Protocole, identique pour tous les essais :**

- découpage stratifié 80 % / 20 %, avec la graine 42 ;
- toutes les comparaisons se font sur des **probabilités out-of-fold (OOF)** du jeu
  d'entraînement, obtenues par validation croisée à 5 plis : chaque client est prédit
  par un modèle entraîné sans lui ;
- le **jeu de test n'est pas utilisé** dans ce notebook. Il ne sert qu'une fois, pour
  évaluer le modèle final (notebook `03_final_model`).

Les modèles candidats sont décrits par des fichiers de configuration dans `configs/`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import PrecisionRecallDisplay

from bank_marketing.application.preparation import prepare_train_test
from bank_marketing.application.train import compute_config_oof_probabilities
from bank_marketing.domain.evaluation import compute_metrics, summarize_thresholds
from bank_marketing.domain.modeling import (
    build_pipeline,
    compute_oof_probabilities,
    create_cross_validation,
)
from bank_marketing.domain.preprocessing import select_features
from bank_marketing.infrastructure.config import load_config
from bank_marketing.infrastructure.figures import save_figure
from bank_marketing.settings import CONFIGS_DIR

sns.set_theme(style="whitegrid")

In [ ]:
final_config = load_config(CONFIGS_DIR / "config.yaml")
RANDOM_STATE = final_config["random_state"]

x_train, _, y_train, _ = prepare_train_test(final_config)
cross_validation = create_cross_validation(n_splits=5, random_state=RANDOM_STATE)

print(f"Jeu d'entraînement : {x_train.shape[0]} clients, {x_train.shape[1]} variables")
print(f"Taux de souscription : {y_train.mean():.2%}")

## 1. Baseline : régression logistique et déséquilibre de la cible

On compare une régression logistique standard à une régression logistique avec
`class_weight="balanced"`, qui donne plus de poids aux souscripteurs minoritaires.
Les métriques sont calculées au seuil par défaut de 0,5.

In [ ]:
def evaluate_oof(features, model_name, params, threshold=0.5):
    """Calcule les métriques OOF d'un modèle à un seuil donné."""
    pipeline = build_pipeline(features, model_name, params, RANDOM_STATE)
    probabilities = compute_oof_probabilities(pipeline, features, y_train, cross_validation)
    return compute_metrics(y_train, probabilities, threshold)


baseline_comparison = pd.DataFrame(
    {
        "logistique_standard": evaluate_oof(x_train, "logistic_regression", {"max_iter": 2000}),
        "logistique_equilibree": evaluate_oof(
            x_train, "logistic_regression", {"max_iter": 2000, "class_weight": "balanced"}
        ),
    }
).T.round(3)
baseline_comparison

Le ROC-AUC et le PR-AUC sont presque identiques : les deux modèles **classent** les
clients de la même façon. En revanche, au seuil de 0,5, le modèle équilibré détecte
beaucoup plus de souscripteurs (rappel nettement supérieur) au prix d'une précision
plus faible, et obtient un meilleur F1.

**Décision :** `class_weight="balanced"` est retenu pour tous les modèles.

## 2. Jeu de variables : complet ou interprétable

Le jeu **interprétable** ne garde qu'une variable macroéconomique (`euribor3m`) et
retire `day_of_week`, peu informative d'après l'EDA.

In [ ]:
balanced_params = {"max_iter": 2000, "class_weight": "balanced"}

feature_set_comparison = pd.DataFrame(
    {
        feature_set: evaluate_oof(
            select_features(x_train, feature_set), "logistic_regression", balanced_params
        )
        for feature_set in ["complete", "interpretable"]
    }
).T.round(3)
feature_set_comparison

Les ROC-AUC sont très proches, mais le jeu **complet** obtient un meilleur PR-AUC et
un F1 nettement supérieur au seuil de 0,5 (0,424 contre 0,381) ; il est retenu pour
la suite. Le jeu
interprétable reste utile pour l'analyse statistique des effets (notebook
`03_final_model`), où la redondance des variables macroéconomiques gênerait
l'estimation.

## 3. Comparaison des trois familles de modèles

Chaque candidat est décrit par un fichier de configuration. Leurs hyperparamètres
proviennent de l'exploration (réglage par Optuna pour la régression logistique et la
forêt aléatoire, en maximisant le PR-AUC en validation croisée).

| Configuration | Modèle |
|---|---|
| `config.yaml` | Régression logistique (elastic net) |
| `random_forest.yaml` | Forêt aléatoire (arbres peu profonds) |
| `svm.yaml` | SVM à noyau RBF, calibré |

In [ ]:
candidates = {
    "Logistique (config.yaml)": "config.yaml",
    "Forêt aléatoire (random_forest.yaml)": "random_forest.yaml",
    "SVM calibré (svm.yaml)": "svm.yaml",
}

oof_probabilities = {
    name: compute_config_oof_probabilities(load_config(CONFIGS_DIR / file), x_train, y_train)
    for name, file in candidates.items()
}

model_comparison = pd.concat(
    [
        summarize_thresholds(name, y_train, probabilities)
        for name, probabilities in oof_probabilities.items()
    ],
    ignore_index=True,
)
model_comparison

**Lecture du tableau :**

- Le **seuil de 0,5 n'a pas le même sens pour tous les modèles**. `class_weight="balanced"`
  gonfle les probabilités de la logistique et de la forêt, alors que le calibrage du SVM
  les ramène vers le taux réel de souscription (≈ 11 %) : au seuil de 0,5, le SVM ne
  détecte presque personne.
- La comparaison équitable se fait donc avec des **métriques indépendantes du seuil**
  (ROC-AUC, PR-AUC) et **au seuil optimisé propre à chaque modèle** (lignes « F1 optimal »).
- Le seuil de **Youden** maximise sensibilité + spécificité − 1 ; il favorise en général
  le rappel.

In [ ]:
figure, axis = plt.subplots(figsize=(7, 5))
for name, probabilities in oof_probabilities.items():
    PrecisionRecallDisplay.from_predictions(y_train, probabilities, name=name, ax=axis)
axis.axhline(y_train.mean(), linestyle="--", color="grey", label="Ciblage au hasard")
axis.set_title("Courbes précision-rappel (probabilités OOF)")
axis.legend(loc="upper right")
save_figure(figure, "courbes_precision_rappel")
plt.show()

La courbe précision-rappel montre, pour chaque niveau de rappel, la précision obtenue.
La ligne pointillée correspond à un ciblage au hasard (précision = taux de
souscription). Plus une courbe est haute, plus le modèle concentre les souscripteurs
parmi les clients qu'il recommande d'appeler.

## 4. Choix du modèle final

| Critère | Logistique | Forêt aléatoire | SVM calibré |
|---|---|---|---|
| PR-AUC (OOF) | 0,423 | 0,436 | 0,345 |
| F1 au seuil optimal (OOF) | 0,487 (seuil 0,589) | 0,484 (seuil 0,639) | 0,473 (seuil 0,234) |
| Interprétabilité | coefficients lisibles | faible | très faible |
| Coût d'entraînement | faible | moyen | élevé |

- Le **SVM est écarté** : son PR-AUC est nettement plus faible et c'est le modèle le plus
  coûteux.
- La **forêt aléatoire** classe légèrement mieux les clients (PR-AUC 0,436 contre
  0,423), mais son F1 au seuil optimal est équivalent, et l'écart est faible au regard
  de la variabilité entre plis.
- La **régression logistique est retenue** pour son interprétabilité : ses coefficients
  s'expliquent directement en termes métier, et ses performances sont équivalentes.

**Modèle final :** régression logistique équilibrée, jeu de variables complet, seuil
choisi par maximisation du F1 sur les probabilités OOF (`configs/config.yaml`).
Ce choix est fait **avant** toute utilisation du jeu de test.